# 07 — Article Tables and Figures

Generates all publication-ready outputs:
- Table 1: Source inventory summary
- Table 2: Layer catalog summary
- Table 3: District-level indicators (LaTeX)
- Figure 1: Priority index choropleth
- Figure 2: Indicator distribution boxplots
- Mermaid pipeline flowchart

In [ ]:
from __future__ import annotations

import pathlib
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from gdynia_thermal_audit.settings import Settings
from gdynia_thermal_audit.logging_utils import setup_logging
from gdynia_thermal_audit.reporting.summary_tables import make_source_inventory_table, make_indicator_summary_table
from gdynia_thermal_audit.reporting.export_article_tables import export_all_article_tables
from gdynia_thermal_audit.reporting.mermaid_flowchart import get_pipeline_flowchart
from gdynia_thermal_audit.reporting.figures import plot_indicators_map
from gdynia_thermal_audit.spatial_units.districts import get_gdynia_districts_placeholder
from gdynia_thermal_audit.utils.io import ensure_dir

setup_logging()
settings = Settings()
ensure_dir('outputs/tables')
ensure_dir('outputs/figures')

In [ ]:
# Load available data (demo fallback)
try:
    df_source = pd.read_csv('data/interim/source_inventory.csv')
except FileNotFoundError:
    df_source = pd.read_csv('data/demo/demo_source_inventory.csv')
    print('Using demo source inventory')

try:
    df_indicators = pd.read_csv('data/processed/indicators_districts.csv')
except FileNotFoundError:
    # Generate minimal placeholder indicators for demo
    gdf = get_gdynia_districts_placeholder()
    df_indicators = pd.DataFrame({
        'unit_id': gdf['district_id'].tolist(),
        'anomaly_count': [5, 12, 3, 8],
        'anomaly_density_per_ha': [0.12, 0.31, 0.08, 0.22],
        'priority_index': [0.4, 0.9, 0.1, 0.6],
        'data_source': ['annotation'] * 4,
    })
    print('Using synthetic indicator demo data')

In [ ]:
# Table 1: Source inventory summary
df_t1 = make_source_inventory_table(df_source)
print('Table 1 — Source Inventory')
df_t1

In [ ]:
# Table 3: Indicator summary
df_t3 = make_indicator_summary_table(df_indicators)
print('Table 3 — Indicator Summary')
df_t3

In [ ]:
# Export all article tables
export_all_article_tables(df_indicators, df_source, output_dir=pathlib.Path('outputs/tables'))
print('Article tables exported to outputs/tables/')

In [ ]:
# Display pipeline flowchart (Mermaid source)
print(get_pipeline_flowchart())